In [1]:
# %% Cell 1 — 安裝套件
!pip install "transformers>=4.30.0" torch numpy scikit-learn tqdm sentencepiece

In [2]:
# %% Cell 2 — 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# %% Cell 3 — 確認去重資料（clean_data）
import os

CLEAN_DIR = '/content/drive/MyDrive/CAIL2018_Lawformer/clean_data'
os.makedirs('/content/drive/MyDrive/CAIL2018_Lawformer/checkpoints', exist_ok=True)

print("確認去重資料...")
for fname in ('data_train.json', 'data_valid.json', 'data_test.json'):
    p = f'{CLEAN_DIR}/{fname}'
    if os.path.exists(p):
        mb = os.path.getsize(p) / 1e6
        print(f"  {fname}: {mb:.1f} MB")
    else:
        print(f"  !! 找不到 {fname}，請確認 clean_data 資料夾")


確認去重資料...
  data_train.json: 241.7 MB
  data_valid.json: 23.8 MB
  data_test.json: 45.0 MB


In [4]:
# %% Cell 4 — Imports
import json
import torch
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    BertModel,
    get_linear_schedule_with_warmup,
)
try:
    from torch.optim import AdamW
except ImportError:
    from transformers import AdamW
from sklearn.metrics import f1_score
from tqdm.notebook import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND'}")

PyTorch: 2.11.0+cu128
GPU: Tesla T4


In [5]:
# %% Cell 5 — Config（全開 fine-tune）
class Config:
    model_name   = "hfl/chinese-macbert-base"
    max_length   = 512
    batch_size   = 8           # fine-tune 較吃記憶體，調小
    grad_accum   = 2           # 等效 batch = 16
    epochs       = 1           # 全開 fine-tune 1 epoch
    encoder_lr   = 2e-5        # encoder 小 lr（保留預訓練語意）
    head_lr      = 1e-3        # 分類頭大 lr
    warmup_ratio = 0.06
    dropout      = 0.1
    threshold    = 0.5
    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data_dir     = "/content/drive/MyDrive/CAIL2018_Lawformer/clean_data"
    save_dir     = "/content/drive/MyDrive/CAIL2018_Lawformer"
    min_fact_len    = 55
    min_label_freq  = 50

cfg = Config()
print(f"Device: {cfg.device}")


Device: cuda


In [6]:
# %% Cell 6 — 讀資料 & 過濾 & 建 label vocab（統一條件）
def load_jsonl(path):
    items = []
    with open(path, encoding='utf-8') as f:
        content = f.read().strip()
    if content.startswith('['):
        items = json.loads(content)
    else:
        for line in content.splitlines():
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

def get_fact_and_charges(item):
    fact = item.get('fact', '')
    if 'meta' in item:
        charges = item['meta'].get('accusation', [])
    else:
        charges = item.get('accusation', [])
    if isinstance(charges, str):
        charges = [charges]
    return fact, charges

print("Loading data...")
train_data = load_jsonl(f'{cfg.data_dir}/data_train.json')
valid_data = load_jsonl(f'{cfg.data_dir}/data_valid.json')
test_data  = load_jsonl(f'{cfg.data_dir}/data_test.json')
print(f"  原始: train={len(train_data):,}  valid={len(valid_data):,}  test={len(test_data):,}")

# === 過濾條件一：法律事實字數 > 55 ===
def filter_short(data):
    return [it for it in data if len(get_fact_and_charges(it)[0]) > cfg.min_fact_len]

train_data = filter_short(train_data)
valid_data = filter_short(valid_data)
test_data  = filter_short(test_data)

# === 過濾條件二：只保留 train 中出現次數 > 50 的罪名 ===
from collections import Counter
charge_counter = Counter()
for it in train_data:
    for c in get_fact_and_charges(it)[1]:
        charge_counter[c] += 1

valid_charges = {c for c, cnt in charge_counter.items() if cnt > cfg.min_label_freq}
print(f"  罪名過濾: {len(charge_counter)} -> {len(valid_charges)} 種（出現 > {cfg.min_label_freq} 次）")

# 建立 vocab（固定順序）
charge2id = {c: i for i, c in enumerate(sorted(valid_charges))}
num_charges = len(charge2id)

# 移除標籤全部被過濾掉的案件（過濾後沒有任何有效罪名的）
def has_valid_charge(item):
    return any(c in charge2id for c in get_fact_and_charges(item)[1])

train_data = [it for it in train_data if has_valid_charge(it)]
valid_data = [it for it in valid_data if has_valid_charge(it)]
test_data  = [it for it in test_data if has_valid_charge(it)]

print(f"  過濾後: train={len(train_data):,}  valid={len(valid_data):,}  test={len(test_data):,}")
print(f"  罪名種類: {num_charges}")

import os
os.makedirs(cfg.save_dir, exist_ok=True)
with open(f'{cfg.save_dir}/charge2id.json', 'w', encoding='utf-8') as f:
    json.dump(charge2id, f, ensure_ascii=False, indent=2)


Loading data...
  原始: train=151,254  valid=16,639  test=31,273
  罪名過濾: 202 -> 160 種（出現 > 50 次）
  過濾後: train=149,879  valid=16,480  test=31,008
  罪名種類: 160


In [7]:
# %% Cell 7 — Dataset
class CAIL2018ChargeDataset(Dataset):
    def __init__(self, data, tokenizer, charge2id, max_length):
        self.data      = data
        self.tokenizer = tokenizer
        self.charge2id = charge2id
        self.max_len   = max_length
        self.n_cls     = len(charge2id)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        fact, charges = get_fact_and_charges(self.data[idx])

        enc = self.tokenizer(
            fact,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        label_vec = torch.zeros(self.n_cls)
        for c in charges:
            if c in self.charge2id:
                label_vec[self.charge2id[c]] = 1.0

        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         label_vec,
        }

In [8]:
# %% Cell 8 — 載入 Tokenizer & Model（全開 fine-tune）
print(f"Loading tokenizer from {cfg.model_name} ...")
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

class ChargeClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1):
        super().__init__()
        self.encoder = BertModel.from_pretrained(model_name)
        # 全開 fine-tune：不凍結任何參數
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))

print(f"Loading model from {cfg.model_name} ...")
model = ChargeClassifier(cfg.model_name, num_charges, cfg.dropout)
model = model.to(cfg.device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"  Trainable params: {trainable/1e6:.1f}M / Total: {total/1e6:.1f}M（全開）")


Loading tokenizer from hfl/chinese-macbert-base ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

...

Loading model from hfl/chinese-macbert-base ...


pytorch_model.bin:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Trainable params: 0.1M / Total: 102.4M


In [9]:
# %% Cell 9 — DataLoaders
train_ds = CAIL2018ChargeDataset(train_data, tokenizer, charge2id, cfg.max_length)
valid_ds = CAIL2018ChargeDataset(valid_data, tokenizer, charge2id, cfg.max_length)
test_ds  = CAIL2018ChargeDataset(test_data,  tokenizer, charge2id, cfg.max_length)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
print(f"train batches={len(train_loader)}  valid={len(valid_loader)}  test={len(test_loader)}")


In [10]:
# %% Cell 10 — Optimizer & Scheduler + pos_weight（差異化 lr）
import torch
from transformers import get_cosine_schedule_with_warmup

# pos_weight：從原始 labels 統計
pos_counts = torch.zeros(num_charges)
for item in train_data:
    _, charges = get_fact_and_charges(item)
    for c in charges:
        if c in charge2id:
            pos_counts[charge2id[c]] += 1
neg_counts = len(train_data) - pos_counts
pos_weight = torch.log1p(neg_counts / (pos_counts + 1e-5)).to(cfg.device)
print(f"pos_weight 範圍: {pos_weight.min():.2f} ~ {pos_weight.max():.2f}")

# 差異化學習率：encoder 用小 lr，分類頭用大 lr
optimizer = AdamW([
    {"params": model.encoder.parameters(),    "lr": cfg.encoder_lr},
    {"params": model.classifier.parameters(), "lr": cfg.head_lr},
], weight_decay=0.01)

total_update_steps = len(train_loader) * cfg.epochs // cfg.grad_accum
warmup_steps       = int(total_update_steps * cfg.warmup_ratio)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_update_steps)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Total update steps: {total_update_steps}  |  Warmup steps: {warmup_steps}")


Pre-computing embeddings (train / valid / test)...


Computing train_emb.pt:   0%|          | 0/4684 [00:00<?, ?it/s]

  Saved: /content/drive/MyDrive/CAIL2018_Lawformer/embeddings_unified/train_emb.pt


Computing valid_emb.pt:   0%|          | 0/515 [00:00<?, ?it/s]

  Saved: /content/drive/MyDrive/CAIL2018_Lawformer/embeddings_unified/valid_emb.pt


Computing test_emb.pt:   0%|          | 0/969 [00:00<?, ?it/s]

  Saved: /content/drive/MyDrive/CAIL2018_Lawformer/embeddings_unified/test_emb.pt
Done!


In [11]:
# %% Cell 11 — Evaluation helper
def evaluate(model, loader, device, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(loader, desc="  Eval", leave=False):
            ids   = batch["input_ids"].to(device)
            amask = batch["attention_mask"].to(device)
            lbls  = batch["labels"].to(device)
            logits = model(ids, amask)
            total_loss += criterion(logits, lbls).item()
            preds = (torch.sigmoid(logits) >= threshold).cpu().numpy().astype(int)
            all_preds.append(preds)
            all_labels.append(lbls.cpu().numpy().astype(int))
    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / len(loader), micro_f1, macro_f1


train=149879  valid=16480  test=31008


In [12]:
# %% Cell 12 — Training Loop（全開 fine-tune）
best_micro_f1  = 0.0
best_ckpt_path = f"{cfg.save_dir}/checkpoints/best_finetune_model.pt"
history        = []

for epoch in range(cfg.epochs):
    model.train()
    optimizer.zero_grad()
    running_loss = 0.0

    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f"Epoch {epoch+1}/{cfg.epochs}")

    for step, batch in pbar:
        ids   = batch["input_ids"].to(cfg.device)
        amask = batch["attention_mask"].to(cfg.device)
        lbls  = batch["labels"].to(cfg.device)

        logits = model(ids, amask)
        loss   = criterion(logits, lbls)
        if cfg.grad_accum > 1:
            loss = loss / cfg.grad_accum
        loss.backward()

        if (step + 1) % cfg.grad_accum == 0:
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        running_loss += loss.item() * cfg.grad_accum
        pbar.set_postfix(loss=f"{running_loss/(step+1):.4f}")

    val_loss, micro_f1, macro_f1 = evaluate(model, valid_loader, cfg.device, cfg.threshold)
    history.append({"epoch": epoch+1, "val_loss": val_loss, "micro_f1": micro_f1, "macro_f1": macro_f1})
    print(f"Epoch {epoch+1} | val_loss={val_loss:.4f} | micro_F1={micro_f1:.4f} | macro_F1={macro_f1:.4f}")

    if micro_f1 > best_micro_f1:
        best_micro_f1 = micro_f1
        torch.save({
            "epoch": epoch + 1, "model_state_dict": model.state_dict(),
            "micro_f1": micro_f1, "macro_f1": macro_f1,
            "charge2id": charge2id, "cfg": {k:str(v) for k,v in vars(cfg).items()},
        }, best_ckpt_path)
        print(f"  Saved best model -> {best_ckpt_path}")


FC model trainable params: 237,984


In [13]:
# %% Cell 13 — Test Evaluation
print("\n=== Loading best checkpoint for test ===")
ckpt = torch.load(best_ckpt_path, map_location=cfg.device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"  Best epoch: {ckpt['epoch']}  |  val micro_F1: {ckpt['micro_f1']:.4f}")

test_loss, test_micro_f1, test_macro_f1 = evaluate(model, test_loader, cfg.device, cfg.threshold)
print(f"\n{'='*45}")
print(f"  TEST  micro-F1 : {test_micro_f1:.4f}")
print(f"  TEST  macro-F1 : {test_macro_f1:.4f}")
print(f"{'='*45}")


pos_weight 範圍: 2.74 ~ 7.91
Total update steps: 234200  |  Warmup steps: 14052


In [14]:
# %% Cell 14 — 8 個評估指標
from sklearn.metrics import accuracy_score, hamming_loss, precision_score, recall_score, f1_score
import numpy as np

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="收集預測"):
        ids   = batch["input_ids"].to(cfg.device)
        amask = batch["attention_mask"].to(cfg.device)
        preds = (torch.sigmoid(model(ids, amask)) >= cfg.threshold).cpu().numpy().astype(int)
        all_preds.append(preds)
        all_labels.append(batch["labels"].numpy().astype(int))

y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

print("="*50)
print(f"  Subset Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"  Hamming Loss    : {hamming_loss(y_true, y_pred):.4f}")
print(f"  micro-Precision : {precision_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
print(f"  micro-Recall    : {recall_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
print(f"  micro-F1        : {f1_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
print(f"  macro-Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"  macro-Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"  macro-F1        : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print("="*50)
